# Module 7: Agent as a Tool

Apply **Pattern 5**: wrap specialized agents as callable `@tool` functions so an orchestrator can delegate like a manager to experts.

![Agent-as-Tool: Orchestrator delegates to Research Agent, Finance Agent, Writer Agent, Legal Agent — each wrapped as @tool](./architecture.png)

**When to use this pattern:**
- Clear hierarchy: one coordinator, many experts
- Each specialist needs its own tools and prompt
- Add / remove specialists without touching the orchestrator

**Key Strands primitive:** `@tool` decorator wrapping an `Agent`.  
The **docstring** is the routing logic — the orchestrator reads it to decide when and how to call each specialist.

**Prerequisites:** Modules 1–6. This module reuses tools from Module 2.

## Components in This Module

| Component | Type | What it does |
|-----------|------|-------------|
| `research_agent` | `@tool` wrapping Agent | Gathers market data, company metrics, competitive intelligence |
| `finance_agent` | `@tool` wrapping Agent | Analyzes financial viability, ROI, unit economics |
| `writer_agent` | `@tool` wrapping Agent | Produces the final investment memo |
| `legal_agent` | `@tool` wrapping Agent | Reviews compliance risks and legal considerations |
| `orchestrator` | `Agent(tools=[...])` | Coordinates the 4 specialists; LLM decides routing |

> **The docstring IS the routing logic.** The orchestrator reads each tool's docstring to decide when to call it and what arguments to pass. Write docstrings for the model, not for developers.

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4, Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")

✅ Setup complete!


In [3]:
import sys, os, time, json
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))

from strands import Agent, tool
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


---

## Part 1: System Prompts

Each specialist agent has a **narrow, focused system prompt**: it does one job and nothing else.
This is what makes the pattern composable: swap a specialist by changing its system prompt.

In [4]:
RESEARCH_PROMPT = (
    "You are a market research specialist. Use your tools to gather company data, "
    "industry benchmarks, and competitive intelligence. Return structured findings: data only."
)

FINANCE_PROMPT = (
    "You are a financial analyst. Analyze the investment brief and market data provided. "
    "Return: revenue projections, unit economics (CAC, LTV, payback period), "
    "ROI estimate, capital efficiency, and a financial verdict (Invest / Invest with conditions / Pass). "
    "Be specific with numbers. 200 words max."
)

WRITER_PROMPT = (
    "You are an investment memo writer. Produce a professional investment analysis memo:\n"
    "## Executive Summary (recommendation in one sentence)\n"
    "## Market Opportunity (size, growth, competitive position)\n"
    "## Financial Highlights (key metrics, ROI, projections)\n"
    "## Risk Assessment (top 3 risks with mitigations)\n"
    "## Recommendation (Invest / Pass, terms, conditions)\n"
    "Under 450 words. Be direct."
)

LEGAL_PROMPT = (
    "You are a legal and compliance reviewer. Review the investment brief for: "
    "regulatory risks, data privacy concerns (GDPR, CCPA), contractual obligations, "
    "IP considerations, and any red flags for due diligence. "
    "Return a bullet-point list of legal risks with severity (High/Medium/Low). 150 words max."
)

ORCHESTRATOR_PROMPT = (
    "You are an investment committee coordinator. For each investment request:\n"
    "1. Call research_agent to gather company and market data.\n"
    "2. Call finance_agent with the brief and research findings to get financial analysis.\n"
    "3. Call legal_agent with the brief to identify legal and compliance risks.\n"
    "4. Call writer_agent with all findings to produce the final investment memo.\n"
    "Execute all four steps. Pass relevant context from each specialist to the next."
)

---

## Part 2: Wrap Specialists as `@tool`

The `@tool` decorator turns each agent into a callable tool. The orchestrator treats them exactly like any other tool: it reads the docstring to decide when and how to call each one.

**Three ways to use agents as tools in Strands:**

```python
# Option A: @tool decorator (most control, multi-parameter)
@tool
def researcher_agent(topic: str) -> str: ...

# Option B: pass Agent directly in tools[] (simplest, single input)
orchestrator = Agent(tools=[researcher_agent_instance, ...])

# Option C: .as_tool() (custom name/description, optional preserve_context)
orchestrator = Agent(tools=[researcher.as_tool(name="...", description="...")])
```

This module uses **Option A**: the `@tool` decorator gives us multi-parameter tools so the orchestrator can pass precise arguments (e.g., option name + description + research context) to the analyzer.

In [5]:
@tool
def research_agent(topic: str) -> str:
    """Gather market data, company metrics, and competitive intelligence for an investment topic.

    Args:
        topic: The company or investment topic to research
    """
    worker = Agent(
        tools=[get_company_data, get_market_benchmarks, get_competitor_data],
        system_prompt=RESEARCH_PROMPT,
        callback_handler=None,
    )
    return str(worker(topic))


@tool
def finance_agent(brief: str, research_context: str) -> str:
    """Analyze financial viability: ROI, unit economics, projections, and investment verdict.

    Args:
        brief: The original investment brief
        research_context: Market and company data from research_agent
    """
    worker = Agent(system_prompt=FINANCE_PROMPT, callback_handler=None)
    return str(worker(f"Investment brief:\n{brief}\n\nMarket research:\n{research_context}"))


@tool
def legal_agent(brief: str) -> str:
    """Review legal and compliance risks: regulatory exposure, data privacy, IP, due diligence flags.

    Args:
        brief: The investment brief to review
    """
    worker = Agent(system_prompt=LEGAL_PROMPT, callback_handler=None)
    return str(worker(brief))


@tool
def writer_agent(brief: str, research_context: str, financial_analysis: str, legal_review: str) -> str:
    """Write the final investment memo synthesizing all specialist findings.
    Call this LAST, after research_agent, finance_agent, and legal_agent.

    Args:
        brief: The original investment brief
        research_context: Findings from research_agent
        financial_analysis: Analysis from finance_agent
        legal_review: Risk review from legal_agent
    """
    worker = Agent(system_prompt=WRITER_PROMPT)
    return str(worker(
        f"Brief:\n{brief}\n\n"
        f"Research:\n{research_context}\n\n"
        f"Financial analysis:\n{financial_analysis}\n\n"
        f"Legal review:\n{legal_review}"
    ))

---

## Part 3: Build and Run the Orchestrator

The orchestrator is a standard `Agent`: but instead of business tools (like `get_company_data`), its tools are other agents. The LLM decides the routing, argument construction, and order of calls.

In [6]:
orchestrator = Agent(
    tools=[research_agent, finance_agent, legal_agent, writer_agent],
    system_prompt=ORCHESTRATOR_PROMPT,
)

INVESTMENT_BRIEF = '''
INVESTMENT BRIEF: NovaCart — Premium Subscription Tier

Company: NovaCart (e-commerce platform, 2M active users, mid-market)
Proposal: Launch a premium subscription tier (Project Nova)
Investment ask: $2M (engineering + marketing)
Expected return: +15% Customer Lifetime Value within 6 months
Options under consideration:
  - Option A: Invite-only exclusive tier ($19.99/mo, top 10% of spenders)
  - Option B: Gradual rollout (5% A/B pilot, $14.99/mo, kill-switch)
  - Option C: Full market launch ($12.99/mo + 30-day free trial)

Produce an investment analysis covering market research, financial viability,
legal risks, and a final recommendation memo.
'''

t0 = time.time()
result = orchestrator(INVESTMENT_BRIEF)
elapsed = time.time() - t0

Running orchestrator (LLM delegates to Research, Finance, Legal, Writer)...


I'll coordinate all four specialist

 agents to produce a comprehensive investment analysis. Let me kick off the research,

 financial analysis, and legal review simultaneously — then synthesize everything

 into a final memo.

**Step 1–3

...

 certainty, churn protection, and legal defensibility**

. The path to Option C's larger up

side remains open — but it must

 be earned through a validated pilot, not assumed from

 the outset.
Done in 84.8s


---

## Part 4: Inspect What the Orchestrator Decided

Unlike Module 2 where the routing was Python code, here the routing lives in the orchestrator's `agent.messages`. Let's see exactly which tools it called, in what order, and with what parameters.

In [7]:
call_count = 0
for msg in orchestrator.messages:
    for block in msg.get("content", []):
        if "toolUse" in block:
            tu = block["toolUse"]
            call_count += 1
            inp = json.dumps(tu.get("input", {}))

=== ORCHESTRATOR TOOL CALLS ===
  1. research_agent({"topic": "NovaCart premium subscription tier e-commerce platform mid-market com...)
  2. finance_agent({"brief": "INVESTMENT BRIEF: NovaCart \u2014 Premium Subscription Tier. Company:...)
  3. legal_agent({"brief": "INVESTMENT BRIEF: NovaCart \u2014 Premium Subscription Tier. Company:...)
  4. writer_agent({"brief": "INVESTMENT BRIEF: NovaCart \u2014 Premium Subscription Tier. Company:...)

Total tool calls: 4
The orchestrator called all 4 specialists — routing decided by the LLM, not Python code.


In [8]:
# Token usage
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})

tool_usage = summary.get("tool_usage", {})
if tool_usage:
    for name, data in tool_usage.items():
        s = data.get("execution_stats", {})

Metric                    Value
--------------------------------
Input tokens             10,187
Output tokens             3,409
Total tokens             13,596
LLM cycles                    3

Per-tool stats:
  legal_agent: calls=1 | avg_time=7.5s
  finance_agent: calls=1 | avg_time=9.9s
  research_agent: calls=1 | avg_time=17.8s
  writer_agent: calls=1 | avg_time=17.5s


---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` wrapping `Agent` | The docstring is the routing signal — the orchestrator reads it |
| 4 specialists as tools | Research → Finance → Legal → Writer, each with its own prompt |
| LLM routing | No Python code wires the specialists — the orchestrator decides order and arguments |
| `callback_handler=None` | Silent sub-agents; only the final writer streams |
| `result.metrics.tool_usage` | Shows call count and timing per specialist |
| Add/remove specialists | Swap a `@tool` without touching any other agent |

---

## What's Next

**Module 8: Capstone** combines all patterns into a complete system.